# Slice-Aware Scheduling

This notebook demonstrates slice-aware scheduling in Sionna's system-level simulator.
UEs are partitioned into network slices (eMBB and URLLC) using a QFI (QoS Flow Indicator),
and each slice receives a dedicated PRB quota with a per-slice scheduling policy.

**Radio parameters used throughout:**

| Parameter | Value |
|-----------|-------|
| Carrier bandwidth | 100 MHz (FR1) |
| Subcarrier spacing | 30 kHz |
| Slot duration | 0.5 ms |
| Total PRBs | 273 |

We compare three scenarios:
1. **Baseline** — no slicing, standard `PFSchedulerSUMIMO`
2. **Equal split** — 136 PRBs for eMBB (PF) + 137 PRBs for URLLC (priority)
3. **URLLC-favoured** — 55 PRBs eMBB + 218 PRBs URLLC

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from sionna.sys import (
    SliceAwarePFScheduler,
    validate_slice_config,
    get_slice_prb_ranges,
    build_slice_prb_mask,
)

device = 'cpu'

## 1  Simulation helpers

Delay-metric helpers that work directly on the `hist` dictionary returned by
`SystemLevelSimulator`.

In [ ]:
def compute_scheduling_delay(hist, num_slots, num_bs, num_ut_per_sector, slot_duration_ms):
    """Mean inter-scheduling gap per UE (ms)."""
    is_scheduled = ~np.isnan(hist['harq'])  # [num_slots, num_bs, num_ut]
    delays = np.full((num_bs, num_ut_per_sector), np.nan)
    for bs in range(num_bs):
        for ue in range(num_ut_per_sector):
            sched_slots = np.where(is_scheduled[:, bs, ue])[0]
            if len(sched_slots) > 1:
                delays[bs, ue] = np.mean(np.diff(sched_slots)) * slot_duration_ms
    return delays


def compute_transmission_delay(hist, num_slots, num_bs, num_ut_per_sector, slot_duration_ms):
    """Mean transmission delay per UE accounting for HARQ retransmissions (ms)."""
    harq = hist['harq']  # [num_slots, num_bs, num_ut]
    delays = np.full((num_bs, num_ut_per_sector), np.nan)
    for bs in range(num_bs):
        for ue in range(num_ut_per_sector):
            harq_seq = harq[:, bs, ue]
            sched_slots = np.where(~np.isnan(harq_seq))[0]
            if len(sched_slots) == 0:
                continue
            tb_delays, nack_count = [], 0
            for s in sched_slots:
                if harq_seq[s] == 0:
                    nack_count += 1
                elif harq_seq[s] == 1:
                    tb_delays.append((1 + nack_count) * slot_duration_ms)
                    nack_count = 0
            if tb_delays:
                delays[bs, ue] = np.mean(tb_delays)
    return delays


def compute_e2e_delay(hist, num_slots, num_bs, num_ut_per_sector, slot_duration_ms):
    """Estimated E2E round-trip delay per UE (ms, symmetric UL/DL assumption)."""
    sched = compute_scheduling_delay(hist, num_slots, num_bs, num_ut_per_sector, slot_duration_ms)
    tx    = compute_transmission_delay(hist, num_slots, num_bs, num_ut_per_sector, slot_duration_ms)
    one_way = np.nansum([sched, tx], axis=0)
    return 2.0 * one_way, sched, tx

## 2  QFI assignment and slice configurations

**Radio parameters: 100 MHz NR, 30 kHz SCS → 273 PRBs, 0.5 ms slot.**

- **QFI 1** → eMBB slice (proportional-fair scheduling)
- **QFI 9** → URLLC slice (priority / highest-rate-first scheduling)

In [ ]:
# --- 100 MHz NR carrier, 30 kHz SCS ---
# 3GPP TS 38.104 Table 5.3.2-1: 100 MHz at 30 kHz SCS → 273 PRBs
num_freq_res  = 273
num_ofdm_sym  = 14           # OFDM symbols per slot
slot_ms       = 0.5          # slot duration at 30 kHz SCS [ms]
num_ut_per_sector = 10

# QFI assignment: first 5 UEs → eMBB (QFI=1), last 5 → URLLC (QFI=9)
qfi_per_ut = torch.cat([
    torch.full((5,), 1, dtype=torch.int32),
    torch.full((5,), 9, dtype=torch.int32),
])
print("QFI assignment:", qfi_per_ut.tolist())

# --- Slice configurations (must sum to 273 PRBs) ---
sweep_configs = {
    "equal_split": {
        1: {"num_prbs": 136, "policy": "proportional_fair"},
        9: {"num_prbs": 137, "policy": "priority"},
    },
    "favor_urllc": {
        1: {"num_prbs":  55, "policy": "proportional_fair"},
        9: {"num_prbs": 218, "policy": "priority"},
    },
}

for name, cfg in sweep_configs.items():
    validate_slice_config(qfi_per_ut, cfg, num_freq_res)
    print(f"Config '{name}' validated OK "
          f"— PRBs: {cfg[1]['num_prbs']} eMBB + {cfg[9]['num_prbs']} URLLC")

## 3  PRB mask inspection

Visualise which PRBs belong to which slice for the equal-split config
(136 eMBB PRBs followed by 137 URLLC PRBs).

In [ ]:
cfg = sweep_configs["equal_split"]
slice_ranges = get_slice_prb_ranges({q: c['num_prbs'] for q, c in cfg.items()})
print("PRB ranges:", slice_ranges)

mask = build_slice_prb_mask(num_freq_res, qfi_per_ut, slice_ranges)
print(f"PRB mask shape: {mask.shape}  (num_freq_res x num_ut)")

plt.figure(figsize=(12, 3))
plt.imshow(mask.numpy().T, aspect='auto', origin='lower', cmap='Blues')
plt.xlabel("PRB index (0-272)")
plt.ylabel("UE index")
plt.title("PRB allocation mask — equal split (100 MHz, 30 kHz SCS)")
plt.colorbar(label="allowed")
plt.tight_layout()
plt.show()

## 4  Standalone scheduler smoke-test

Sanity check on `SliceAwarePFScheduler` without the full `SystemLevelSimulator`.

In [ ]:
slice_cfg = sweep_configs["equal_split"]

scheduler = SliceAwarePFScheduler(
    qfi_per_ut=qfi_per_ut,
    slice_prb_config=slice_cfg,
    num_ut=num_ut_per_sector,
    num_freq_res=num_freq_res,
    num_ofdm_sym=num_ofdm_sym,
    device=device,
)

rate_last = torch.rand(num_ut_per_sector) * 100
rate_ach  = torch.rand(num_ofdm_sym, num_freq_res, num_ut_per_sector) * 100
is_sched  = scheduler(rate_last, rate_ach)  # [sym, prb, ut, streams]

print(f'is_scheduled shape: {is_sched.shape}')

# Verify slice isolation
is_sched_np = is_sched.squeeze(-1).numpy()  # [sym, prb, ut]
slice_ranges = scheduler.slice_ranges
for qfi_val, (start, end) in slice_ranges.items():
    ut_mask_np = (qfi_per_ut == qfi_val).numpy()
    out_of_slice = is_sched_np[:, start:end, :][:, :, ~ut_mask_np]
    assert not out_of_slice.any(), f'Slice isolation violated for QFI {qfi_val}!'
print("Slice isolation check: PASSED")

# Verify exactly one UE per PRB
per_prb_sum = is_sched.sum(dim=-2).squeeze(-1)
assert (per_prb_sum == 1).all(), 'Not exactly one UE per PRB!'
print("Single-UE-per-PRB check: PASSED")

## 5  Integration with `SystemLevelSimulator`

This cell runs both a **baseline** and a **slice-aware** simulation.

**Prerequisites** — run all cells in `End-to-End_Example.ipynb` first so that
the variables `sls`, `num_slots`, `alpha_ul`, `p0_dbm_ul`, `bler_target`,
and `olla_delta_up` are defined in the kernel.

If those variables are not available, the cell automatically falls back to
synthetic placeholder data so that Sections 6 and 7 still run and produce
illustrative (not physically accurate) results.

> **Note on the resource grid**: the `End-to-End_Example.ipynb` notebook
> currently uses `subcarrier_spacing = 15e3` Hz and `fft_size = 128`.
> To match the 100 MHz / 30 kHz SCS / 273 PRB scenario, change those
> parameters to `subcarrier_spacing = 30e3` and `fft_size = 273` before
> instantiating the `ResourceGrid` and `SystemLevelSimulator`.

In [ ]:
# ==============================================================
# PREREQUISITES: run End-to-End_Example.ipynb first so that
#   sls, num_slots, alpha_ul, p0_dbm_ul, bler_target, olla_delta_up
# are defined in this kernel.
#
# If those variables are not available, the except block below falls
# back to synthetic data so the rest of the notebook still runs.
# ==============================================================

try:
    from sionna.sys import PFSchedulerSUMIMO

    def _to_numpy_hist(h):
        return {k: v.cpu().numpy() for k, v in h.items()}

    # ---- Baseline run (standard PF, no slicing) ----
    sls.scheduler = PFSchedulerSUMIMO(
        sls.num_ut_per_sector,
        sls.resource_grid.fft_size,
        sls.resource_grid.num_ofdm_symbols,
        batch_size=[sls.batch_size, sls.num_bs],
        num_streams_per_ut=sls.num_streams_per_ut,
        precision=sls.precision,
        device=sls.device,
    )
    print('Running baseline simulation...')
    hist_baseline = _to_numpy_hist(
        sls(num_slots, alpha_ul, p0_dbm_ul, bler_target, olla_delta_up))

    # Derive slot duration from the resource grid (should be 0.5 ms for 30 kHz SCS)
    _slot_ms  = float(sls.slot_duration * 1e3)
    _num_bs   = sls.num_bs
    _num_slots = num_slots
    print(f'  slot_duration = {_slot_ms:.4f} ms  (expect 0.5 ms for 30 kHz SCS)')

    # ---- Slice-aware run (equal 136 / 137 split) ----
    # Remap slice config to actual fft_size in case it differs from num_freq_res
    _fft = sls.resource_grid.fft_size
    n9 = _fft // 2 + _fft % 2   # URLLC gets the ceiling half
    n1 = _fft - n9
    slice_cfg_run = {
        1: {'num_prbs': n1, 'policy': 'proportional_fair'},
        9: {'num_prbs': n9, 'policy': 'priority'},
    }
    validate_slice_config(qfi_per_ut, slice_cfg_run, _fft)

    sls.scheduler = SliceAwarePFScheduler(
        qfi_per_ut=qfi_per_ut,
        slice_prb_config=slice_cfg_run,
        num_ut=sls.num_ut_per_sector,
        num_freq_res=_fft,
        num_ofdm_sym=sls.resource_grid.num_ofdm_symbols,
        batch_size=[sls.batch_size, sls.num_bs],
        num_streams_per_ut=sls.num_streams_per_ut,
        precision=sls.precision,
        device=sls.device,
    )
    print('Running slice-aware simulation...')
    hist_sliced = _to_numpy_hist(
        sls(num_slots, alpha_ul, p0_dbm_ul, bler_target, olla_delta_up))

    _using_real_hist = True
    print('Both simulations complete.')

except NameError as e:
    # ---- Synthetic fallback ----
    print(f'SystemLevelSimulator not found ({e}).')
    print('Falling back to synthetic placeholder data.')
    print('  slot_duration = 0.5 ms  (30 kHz SCS assumed)')

    _num_slots = 400
    _num_bs    = 1
    _slot_ms   = 0.5   # 30 kHz SCS -> 0.5 ms per slot

    _rng = np.random.default_rng(0)

    def _make_hist(harq_prob_per_ut):
        harq = np.full((_num_slots, _num_bs, num_ut_per_sector), np.nan)
        for s in range(_num_slots):
            for u, p in enumerate(harq_prob_per_ut):
                if _rng.random() < p:
                    harq[s, 0, u] = float(_rng.random() > 0.1)
        return {'harq': harq}

    hist_baseline = _make_hist([0.3] * num_ut_per_sector)
    hist_sliced   = _make_hist(
        [0.2, 0.2, 0.2, 0.2, 0.2,   # eMBB
         0.5, 0.5, 0.5, 0.5, 0.5])  # URLLC

    _using_real_hist = False

## 6  KPI computation and comparison

Compute per-slice E2E round-trip delay from `hist_baseline` and `hist_sliced`
produced in Section 5.

In [ ]:
# Compute per-UE delays
e2e_base,   sched_base,   tx_base   = compute_e2e_delay(
    hist_baseline, _num_slots, _num_bs, num_ut_per_sector, _slot_ms)
e2e_sliced, sched_sliced, tx_sliced = compute_e2e_delay(
    hist_sliced, _num_slots, _num_bs, num_ut_per_sector, _slot_ms)

qfi_np = qfi_per_ut.numpy()
slice_names = {1: 'eMBB (QFI=1)', 9: 'URLLC (QFI=9)'}

src_label = 'real simulator' if _using_real_hist else 'synthetic placeholder'
print(f'Results from: {src_label}  |  slot duration = {_slot_ms} ms')
print()
print(f"{'Slice':<20} {'Baseline E2E RTT':>18} {'Slice-aware E2E RTT':>22}")
print("-" * 62)
for qfi_val in sorted(np.unique(qfi_np)):
    ue_mask = qfi_np == qfi_val
    b_mean  = np.nanmean(e2e_base[:, ue_mask])
    s_mean  = np.nanmean(e2e_sliced[:, ue_mask])
    name    = slice_names.get(qfi_val, f'QFI={qfi_val}')
    print(f'{name:<20} {b_mean:>16.2f} ms {s_mean:>20.2f} ms')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
slice_colors = {1: 'steelblue', 9: 'tomato'}

for ax, (scenario_name, e2e) in zip(
        axes, [('Baseline', e2e_base), ('Slice-aware (equal split)', e2e_sliced)]):
    for qfi_val in sorted(np.unique(qfi_np)):
        ue_mask = qfi_np == qfi_val
        vals = e2e[:, ue_mask].ravel()
        vals = vals[~np.isnan(vals)]
        sorted_vals = np.sort(vals)
        cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
        ax.plot(sorted_vals, cdf,
                color=slice_colors[qfi_val],
                label=slice_names.get(qfi_val, f'QFI={qfi_val}'))
    ax.set_title(scenario_name)
    ax.set_xlabel('E2E round-trip delay (ms)')
    ax.set_ylabel('CDF')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('E2E RTT CDF per slice  —  100 MHz / 30 kHz SCS / 0.5 ms slot / 273 PRBs')
plt.tight_layout()
plt.show()

## 7  PRB sweep across configurations

Sweep the URLLC PRB fraction from 20 % to 80 % (of 273 total PRBs) and
plot per-slice mean E2E delay.  When a real simulator is available
(`_using_real_hist = True`), the simulator is re-run for each split;
otherwise the sweep uses the synthetic `_make_hist` model.

In [ ]:
urllc_prb_fractions = [0.2, 0.4, 0.5, 0.6, 0.8]
urllc_mean_e2e = []
embb_mean_e2e  = []

for frac in urllc_prb_fractions:
    n_urllc = max(1, round(num_freq_res * frac))
    n_embb  = num_freq_res - n_urllc
    if n_embb < 1:
        continue

    cfg = {
        1: {'num_prbs': n_embb,  'policy': 'proportional_fair'},
        9: {'num_prbs': n_urllc, 'policy': 'priority'},
    }

    if _using_real_hist:
        sls.scheduler = SliceAwarePFScheduler(
            qfi_per_ut=qfi_per_ut,
            slice_prb_config=cfg,
            num_ut=sls.num_ut_per_sector,
            num_freq_res=sls.resource_grid.fft_size,
            num_ofdm_sym=sls.resource_grid.num_ofdm_symbols,
            batch_size=[sls.batch_size, sls.num_bs],
            num_streams_per_ut=sls.num_streams_per_ut,
            precision=sls.precision,
            device=sls.device,
        )
        h_raw = sls(num_slots, alpha_ul, p0_dbm_ul, bler_target, olla_delta_up)
        h = {k: v.cpu().numpy() for k, v in h_raw.items()}
        ns, nb_run = _num_slots, _num_bs
    else:
        p_urllc = 0.1 + 0.8 * frac
        p_embb  = 0.1 + 0.8 * (1 - frac)
        h = _make_hist([p_embb] * 5 + [p_urllc] * 5)
        ns, nb_run = _num_slots, _num_bs

    e2e, _, _ = compute_e2e_delay(h, ns, nb_run, num_ut_per_sector, _slot_ms)
    urllc_mean_e2e.append(np.nanmean(e2e[:, qfi_np == 9]))
    embb_mean_e2e.append(np.nanmean(e2e[:, qfi_np == 1]))
    print(f'frac={frac:.0%}  eMBB PRBs={n_embb}  URLLC PRBs={n_urllc}  '
          f'->  eMBB E2E={embb_mean_e2e[-1]:.2f} ms  '
          f'URLLC E2E={urllc_mean_e2e[-1]:.2f} ms')

plt.figure(figsize=(7, 4))
plt.plot(urllc_prb_fractions, urllc_mean_e2e, 'o-',  color='tomato',
         label='URLLC (QFI=9)')
plt.plot(urllc_prb_fractions, embb_mean_e2e,  's--', color='steelblue',
         label='eMBB  (QFI=1)')
plt.xlabel('URLLC PRB fraction  (of 273 total PRBs)')
plt.ylabel('Mean E2E RTT (ms)')
plt.title('Per-slice E2E delay vs. PRB allocation\n'
          '100 MHz / 30 kHz SCS / 0.5 ms slot')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

| Component | Location |
|-----------|----------|
| `SliceAwarePFScheduler` | `sionna.sys` |
| `build_slice_prb_mask` | `sionna.sys` |
| `validate_slice_config` | `sionna.sys` |
| `get_slice_prb_ranges` | `sionna.sys` |

**Radio configuration:**

| Parameter | Value |
|-----------|-------|
| Carrier bandwidth | 100 MHz (FR1) |
| Subcarrier spacing | 30 kHz |
| Slot duration | **0.5 ms** |
| Total PRBs | **273** |

**To run with the real simulator:**
1. Open `End-to-End_Example.ipynb`; change `subcarrier_spacing = 30e3` and
   `fft_size = 273` in the resource-grid cell, then run all cells.
2. Return here and run from Section 2 onward.
3. Section 5 detects `sls` and runs the real simulation;
   Sections 6 and 7 then show real per-slice KPIs.